In [ ]:
from villas.dataprocessing.readtools import *
from villas.dataprocessing.timeseries import *
from villas.dataprocessing.timeseries import TimeSeries as ts
import matplotlib.pyplot as plt
import numpy as np
import dpsimpy

# %matplotlib widget

## Voltage Source + Resistor + Shunt

In [ ]:
time_step = 0.00005
final_time = 1

In [ ]:
sim_name = "EMT_VoltageSource_Resistor_Shunt"
dpsimpy.Logger.set_log_dir("logs/" + sim_name)

# Nodes
gnd = dpsimpy.emt.SimNode.gnd
n1 = dpsimpy.emt.SimNode("n1", dpsimpy.PhaseType.ABC)
n2 = dpsimpy.emt.SimNode("n2", dpsimpy.PhaseType.ABC)

# Components
vs = dpsimpy.emt.ph3.VoltageSource("vs1")
vs.set_parameters(
    dpsimpy.Math.single_phase_variable_to_three_phase(complex(100000, 0)), 50
)

res = dpsimpy.emt.ph3.Resistor("R1", dpsimpy.LogLevel.debug)
res.set_parameters(dpsimpy.Math.single_phase_parameter_to_three_phase(5))

shunt = dpsimpy.emt.ph3.Shunt("Shunt1", dpsimpy.LogLevel.debug)
shunt.set_parameters(0.1, 3.14159)

# Topology
vs.connect([gnd, n1])
res.connect([n1, n2])
shunt.connect([n2])

sys = dpsimpy.SystemTopology(50, [n1, n2], [vs, res, shunt])

# Logging
logger = dpsimpy.Logger(sim_name)
logger.log_attribute("v1", "v", n1)
logger.log_attribute("v2", "v", n2)
logger.log_attribute("iR", "i_intf", res)
logger.log_attribute("iShunt", "i_intf", shunt)

# Simulation
sim = dpsimpy.Simulation(sim_name)
sim.set_system(sys)
sim.set_time_step(time_step)
sim.set_final_time(final_time)
sim.set_domain(dpsimpy.Domain.EMT)
sim.add_logger(logger)
sim.run()

In [ ]:
sim_name = "DP_VoltageSource_Resistor_Shunt"
dpsimpy.Logger.set_log_dir("logs/" + sim_name)

# Nodes
gnd = dpsimpy.dp.SimNode.gnd
n1 = dpsimpy.dp.SimNode("n1")
n2 = dpsimpy.dp.SimNode("n2")

# Components
vs = dpsimpy.dp.ph1.VoltageSource("vs1")
vs.set_parameters(complex(100000, 0))

res = dpsimpy.dp.ph1.Resistor("R1", dpsimpy.LogLevel.debug)
res.set_parameters(5)

shunt = dpsimpy.dp.ph1.Shunt("Shunt1", dpsimpy.LogLevel.debug)
shunt.set_parameters(0.1, 3.14159)

# Topology
vs.connect([gnd, n1])
res.connect([n1, n2])
shunt.connect([n2])

sys = dpsimpy.SystemTopology(50, [n1, n2], [vs, res, shunt])

# Logging
logger = dpsimpy.Logger(sim_name)
logger.log_attribute("v1", "v", n1)
logger.log_attribute("v2", "v", n2)
logger.log_attribute("iR", "i_intf", res)
logger.log_attribute("iShunt", "i_intf", shunt)

# Simulation
sim = dpsimpy.Simulation(sim_name)
sim.set_system(sys)
sim.set_time_step(time_step)
sim.set_final_time(final_time)
sim.set_domain(dpsimpy.Domain.DP)
sim.add_logger(logger)
sim.run()

In [ ]:
model_name = "VoltageSource_Resistor_Shunt"

path_DP = "logs/" + "DP_" + model_name + "/"
dpsim_result_file_DP = path_DP + "DP_" + model_name + ".csv"
ts_dpsim_DP = read_timeseries_csv(dpsim_result_file_DP)
ts_dpsim_DP_shifted = ts.frequency_shift_list(ts_dpsim_DP, 50)

path_EMT = "logs/" + "EMT_" + model_name + "/"
dpsim_result_file_EMT = path_EMT + "EMT_" + model_name + ".csv"
ts_dpsim_EMT = read_timeseries_csv(dpsim_result_file_EMT)

### Plot voltages

In [ ]:
plt.figure()

# EMT
var_names = ["v1_0", "v2_0"]
for var_name in var_names:
    plt.plot(
        ts_dpsim_EMT[var_name].time,
        np.sqrt(3 / 2) * ts_dpsim_EMT[var_name].values,
        label=var_name,
    )

# DP
var_names = ["v1_shift", "v2_shift"]
for var_name in var_names:
    plt.plot(
        ts_dpsim_DP_shifted[var_name].time,
        ts_dpsim_DP_shifted[var_name].values,
        label=var_name + " (DP)",
        linestyle="-.",
    )

plt.legend()
plt.show()

### Plot currents

In [ ]:
plt.figure()

# EMT
var_names = ["iR_0", "iShunt_0"]
for var_name in var_names:
    plt.plot(
        ts_dpsim_EMT[var_name].time,
        np.sqrt(3 / 2) * ts_dpsim_EMT[var_name].values,
        label=var_name,
    )

# DP
var_names = ["iR_shift", "iShunt_shift"]
for var_name in var_names:
    plt.plot(
        ts_dpsim_DP_shifted[var_name].time,
        ts_dpsim_DP_shifted[var_name].values,
        label=var_name + " (DP)",
        linestyle="-.",
    )

plt.legend()
plt.show()

### Comparison DP vs. EMT

In [ ]:
plt.figure()
for name in [("v1_0", "v1_shift"), ("v2_0", "v2_shift")]:
    plt.plot(
        ts_dpsim_EMT[name[0]].time,
        np.sqrt(3 / 2) * ts_dpsim_EMT[name[0]].values
        - ts_dpsim_DP_shifted[name[1]].values,
        label=name[0] + " vs. " + name[1],
    )
plt.legend()
plt.show()

In [ ]:
plt.figure()
for name in [("iR_0", "iR_shift"), ("iShunt_0", "iShunt_shift")]:
    plt.plot(
        ts_dpsim_EMT[name[0]].time,
        np.sqrt(3 / 2) * ts_dpsim_EMT[name[0]].values
        - ts_dpsim_DP_shifted[name[1]].values,
        label=name[0] + " vs. " + name[1],
    )
plt.legend()
plt.show()

### Assertion DP vs. EMT

In [ ]:
compare_errors_abs = []
compare_errors_rel = []
for name in [
    ("v1_0", "v1_shift"),
    ("v2_0", "v2_shift"),
    ("iR_0", "iR_shift"),
    ("iShunt_0", "iShunt_shift"),
]:
    compare_errors_abs.append(
        np.absolute(
            np.sqrt(3 / 2) * ts_dpsim_EMT[name[0]].values
            - ts_dpsim_DP_shifted[name[1]].values
        ).max()
    )
    compare_errors_rel.append(
        np.absolute(
            np.sqrt(3 / 2) * ts_dpsim_EMT[name[0]].values
            - ts_dpsim_DP_shifted[name[1]].values
        ).max()
        / ts_dpsim_DP_shifted[name[1]].values.max()
    )
    print(name[0] + " vs. " + name[1] + " (abs): " + str(compare_errors_abs[-1]))
    print(name[0] + " vs. " + name[1] + " (rel): " + str(compare_errors_rel[-1]))
print("Max rel error: " + "{:.2}".format(np.max(compare_errors_rel) * 100) + "%")
assert np.max(compare_errors_rel) < 1e-3